In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.16 Phonons and the Debye Model

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.16",
    title="Phonons and the Debye Model",
    blurb="A crystal is the radiation problem with two mercies: sound has a speed, and a "
    "lattice has a headcount — exactly 3N modes, so no catastrophe is even possible. "
    "We diagonalize the chain with the same matrix that hopped electrons four "
    "notebooks ago, derive at last the T-cubed law this volume has twice used on "
    "trust, recover Dulong–Petit as a limit instead of an assumption, and check the "
    "Debye temperature of copper two ways — by calorimeter and by loudspeaker — "
    "getting the same number to three digits.",
    difficulty="advanced",
    estimate="190–230 min",
)

## Notebook overview

Three promises come due in this notebook, and all three are kept in sound. The confession of [§7.5](oscillator-at-temperature.ipynb) (the Einstein solid collapsing exponentially at 50 K where diamond's measured heat capacity falls only as $T^3$) is resolved. The coefficient $A = 12\pi^4R/5\theta_D^3$ that [§7.10](fermi-gas-finite-temperature.ipynb)
invoked on trust for its $C/T$-versus-$T^2$ plot is derived here and evaluated for copper's
$\theta_D = 343$ K: it returns $0.0482$ mJ mol⁻¹K⁻⁴, the exact figure [§7.10](fermi-gas-finite-temperature.ipynb) displayed, and the
crossover $T^* = 3.2$ K of that notebook now stands fully first-principles. And Volume V's
Dulong–Petit law, asserted there as equipartition, is recovered as a high-temperature *limit*
of an exact quantum expression.

The mechanics comes first, and it is a reunion: the dynamical matrix of an $N$-atom ring of
masses and springs is *the same circulant matrix* that hopped electrons in the tight-binding chain of [§7.12](bloch-theorem-band-structure.ipynb): one matrix, two physics; electrons hop, atoms sway. `numpy.linalg.eigh` returns the
dispersion $\omega(k) = 2\sqrt{K/M}\,|\sin(ka/2)|$ at machine precision, sound emerges as the
small-$k$ slope, and the structural point stands against [§7.14](photon-gas-planck.ipynb): the photon gas had infinitely
many modes and needed $\hbar$ to tame them, while a crystal has **exactly $3N$** and cannot suffer a catastrophe: its ultraviolet cutoff is not a hypothesis but a census. Each mode is
then the oscillator of [§7.5](oscillator-at-temperature.ipynb) with $\mu = 0$ by the unconserved-number argument of [§7.14](photon-gas-planck.ipynb), and the reason
Einstein failed becomes physical before any integral: his solid, all modes at one frequency,
has nothing soft to offer at low temperature, while a real chain's acoustic branch offers modes at *every* softness: the modes with $\hbar\omega \lesssim k_BT$ are awake, their count
grows as $T^d$, and each carries $\sim k_BT$. Debye's model is that census with its constant
computed: linear dispersion cut off at $\omega_D$, the universal curve
$C/3Nk_B = 3t^3\int_0^{1/t}x^4e^x/(e^x-1)^2\,dx$ (nondimensionalized per the standing
rule of [§7.14](photon-gas-planck.ipynb)), Dulong–Petit above, $(4\pi^4/5)\,t^3$ below: the integral of [§7.3](statistical-toolkit.ipynb) in its third starring role.

The data jewel is an over-determination: $\theta_D$ can be measured with a calorimeter,
through $C(T)$ — or with a loudspeaker, through the sound speeds and the atom density, two experiments that share no apparatus. Copper agrees with itself to three digits (342 K by sound, 343 K by heat); aluminum comes in at ratio 0.93; lead at 0.72. The misses are taught, not hidden: lead is soft, heavy, and anharmonic, its springs barely Hookean at
accessible temperatures. Diamond is then closed honestly at both ends: Debye with the low-$T$
$\theta_D = 2230$ K owns the regime where Einstein collapses (the ratio of the two models
falls from 1.5 to 0.002 between 300 and 75 K), yet *undershoots* the measured 300 K heat capacity, because the true phonon spectrum is not a parabola with a cliff, so the fitted
$\theta_D$ drifts with the fitting window, and that drift is itself data. A student exercise
proves that dimension counts (the exact 1D chain's $C/T$ plateaus at exactly $\pi/3$) and meets the finite-size freeze on the way: a 64-atom chain goes silent below its softest mode.

> **Conventions (this notebook).** The chain runs in working units $K = M = a = 1$ (so
> $\omega(k) = 2|\sin(k/2)|$, $v_s = 1$, $k_B = \hbar = 1$); the material data run in SI with
> cited inputs. The reduced temperature is $t = T/\theta_D$; heat capacities are per mode
> ($C/3Nk_B$ for crystals, $C/Nk_B$ for the chain) except where a molar unit is stated. The
> $k = 0$ translation mode is a free coordinate, not an oscillator, and is excluded from every
> thermal sum. Frequencies from `numpy.linalg.eigh` pass through
> `numpy.sqrt(numpy.maximum(w2, 0))` — the zero-mode round-off trap, demonstrated before it is
> clamped. The Debye integral is evaluated by `scipy.integrate.quad` on the nondimensional
> kernel only (the rule of [§7.14](photon-gas-planck.ipynb)), with `numpy.expm1` and a stated lower cutoff.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the `eigh` spectrum against the closed-form dispersion at $10^{-12}$; the
> census estimate against the measured mode count; both Debye limits against their stated
> values; the $A$ coefficient against the digit [§7.10](fermi-gas-finite-temperature.ipynb) displayed; copper's acoustic $\theta_D$
> against the calorimetric one; diamond's model ratios against the cited anchors; the chain's
> $C/T$ against $\pi/3$. A ✓ is strong evidence; a ✗ is a prompt to *locate the discrepancy*.
>
> **Scope.** Real phonon spectra (neutron and X-ray spectroscopy), anharmonicity and thermal
> expansion, phonon transport, graphene's flexural modes, and nanoparticle calorimetry are
> named horizons. See Debye 1912; Einstein 1907 (the story completed here); Ashcroft & Mermin
> (Chs. 22–23); Kittel (Ch. 5). Cross-reference [§7.5](oscillator-at-temperature.ipynb) (the oscillator; the confession), [§7.10](fermi-gas-finite-temperature.ipynb)
> (the coefficient, now derived), [§7.12](bloch-theorem-band-structure.ipynb) (the circulant matrix and the counting), [§7.14](photon-gas-planck.ipynb)
> ($\mu = 0$; the nondimensionalization rule; the catastrophe contrast), [§7.3](statistical-toolkit.ipynb) ($\pi^4/15$,
> third use), [§5.5](../05-classical-stat-mech/ergodicity.ipynb) (Dulong–Petit as equipartition), and forward to [§7.17](bose-einstein-condensation.ipynb) (BEC, where the boson
> number *is* conserved and the ceiling bites).

## Theory in brief

### The chain, diagonalized — one matrix, two physics

$N$ atoms of mass $M$ on a ring, nearest neighbours coupled by springs $K$: Newton gives
$M\ddot u_n = K(u_{n+1} - 2u_n + u_{n-1})$, so normal modes solve the eigenproblem of the
**dynamical matrix**

```{math}
:label: eq-chain-dispersion
D = \frac{K}{M}\left(2I - S - S^{\mathsf T}\right)
\qquad\Longrightarrow\qquad
\omega(k) = 2\sqrt{\frac{K}{M}}\,\left|\sin\frac{ka}{2}\right|,
\qquad
v_s = a\sqrt{K/M},
```

with $S$ the one-site shift. This is, to the letter, the tight-binding circulant of [§7.12](bloch-theorem-band-structure.ipynb). One matrix, two physics: electrons hop through it, atoms sway by it. Two facts carry the notebook.
First, at small $k$ the dispersion is linear: **sound**, with speed $v_s$. Second, the ring
has **exactly $N$ modes** (a 3D crystal, $3N$). Say it against [§7.14](photon-gas-planck.ipynb): the photon gas owned
infinitely many modes and diverged classically; a crystal *cannot*: its ultraviolet cutoff is a census, not a hypothesis, and Debye's $\omega_D$ below is bookkeeping, not an ansatz.

### Quantization, and what a phonon is

Each normal mode is the harmonic oscillator of [§7.5](oscillator-at-temperature.ipynb) (literally, in the mode coordinates), so its thermal energy is

```{math}
:label: eq-phonon-quantization
U = \sum_{\text{modes}} \hbar\omega_k\left(n_B(\hbar\omega_k) + \tfrac12\right),
\qquad
\mu = 0 \text{ (phonon number is not conserved)}.
```

The honest sentence about what a phonon *is*: a bookkeeping quantum of collective lattice motion — not a particle sitting anywhere, but as real as anything that scatters neutrons, which it does (the outward breath: inelastic neutron spectroscopy). The zero-point sum is $(9/8)Nk_B\theta_D$ in the Debye model: real, and visible in isotope effects.

### Why Einstein failed: the census argument

The confession of [§7.5](oscillator-at-temperature.ipynb), recalled with its number: Einstein's all-modes-at-$\omega_E$ solid predicts $C \sim e^{-\theta_E/T}$, which for diamond at 50 K is $\sim 10^{-9}$ of the classical value, while the measurement falls only as $T^3$. The resolution is physical before it is integral:

```{math}
:label: eq-census
N_{\text{awake}}(T) \sim N\left(\frac{T}{\theta}\right)^{d},
\qquad
C \sim N_{\text{awake}}\,k_B \;\propto\; T^{d}.
```

An acoustic branch offers modes at *every* softness. At temperature $T$ the modes with
$\hbar\omega \lesssim k_BT$ are awake and each holds $\sim k_BT$; in $d$ dimensions their
count grows as $(T/\theta)^d$. That is the notebook's actual content; Debye's integral is this census with its constant computed. Einstein's solid fails because it has nothing soft to
offer: below $\theta_E$ its one frequency freezes and the whole solid goes silent at once.

### The Debye model

Keep exactly the low-$\omega$ physics — linear dispersion with the three acoustic
polarizations averaged as $v_D = [3/(v_L^{-3} + 2v_T^{-3})]^{1/3}$ — and cut off where the
census says to stop:

```{math}
:label: eq-debye-model
g(\omega) = \frac{3V\omega^2}{2\pi^2v_D^3},
\quad
\omega_D^3 = 6\pi^2 n\,v_D^3,
\quad
\frac{C}{3Nk_B} = 3t^3\!\int_0^{1/t}\!\frac{x^4e^x}{(e^x-1)^2}\,dx,
\quad t = \frac{T}{\theta_D}.
```

The integral is nondimensionalized per the standing rule of [§7.14](photon-gas-planck.ipynb), with `numpy.expm1` in the kernel.
Both limits close old accounts: at $t \gg 1$ the curve reaches $0.998$ at $t = 5$ (**Dulong–Petit**, Volume V's $3Nk_B$ now a *limit* rather than an assumption), and at
$t \ll 1$ it lands on $(4\pi^4/5)\,t^3$ at ratio 1.00000 by $t = 0.02$, since
$\int_0^\infty x^4e^x/(e^x-1)^2dx = 4\pi^4/15$: the Bose integral of [§7.3](statistical-toolkit.ipynb), in its third starring
role.

### The coefficient 7.10 used, derived

Keep only the $t \ll 1$ limit of the Debye curve, the $(4\pi^4/5)\,t^3$ law just verified,
and restate it per mole ($Nk_B \to R$); the low-temperature coefficient follows with no
free parameter:

```{math}
:label: eq-A-coefficient
C = \frac{12\pi^4}{5}\,Nk_B\,t^3
\;\Longrightarrow\;
A = \frac{12\pi^4 R}{5\,\theta_D^3}
= 0.0482\ \text{mJ mol}^{-1}\text{K}^{-4}
\quad(\text{copper}, \theta_D = 343\ \text{K}),
```

the exact figure [§7.10](fermi-gas-finite-temperature.ipynb) invoked for its $C/T$-versus-$T^2$ plot with the derivation deferred to
this notebook. The cross-reference closes in public: the coefficient invoked six notebooks ago
and the coefficient derived today agree to the displayed digit, and the crossover of [§7.10](fermi-gas-finite-temperature.ipynb)
$T^* = \sqrt{\gamma/A} = 3.2$ K now stands fully derived.

### Two experiments, one number

The census fixes $\omega_D^3 = 6\pi^2 n\,v_D^3$, so $k_B\theta_D = \hbar\omega_D$ is
computable from sound speeds and atom density with no calorimeter in sight; the same number
must independently emerge from fitting the measured $C(T)$:

```{math}
:label: eq-sound-vs-heat
\theta_D^{\text{sound}} = \frac{\hbar v_D}{k_B}\,(6\pi^2 n)^{1/3}
\qquad\text{versus}\qquad
\theta_D^{\text{heat}} \text{ from } C(T).
```

$\theta_D$ is over-determined: calorimetry fixes it through the heat capacity, but acoustics
fixes it through $v_L$, $v_T$, and the atom density — two experiments sharing no apparatus.
Copper: 342 K by sound against 343 K by heat (ratio 1.00: the harmonic approximation earning its keep). Aluminum: 399 vs 428 (0.93). Lead: 76 vs 105 (0.72), and the miss is the lesson:
lead is soft, heavy, and low-$\theta$; its springs are not Hookean at accessible temperatures.
**Anharmonicity** is the model's edge, and thermal expansion is its everyday face (a purely harmonic crystal would not expand at all).

### Diamond, closed honestly

Diamond is where the story began: Einstein's 1907 paper fit its heat capacity, far below
Dulong–Petit at room temperature, and Debye's 1912 density of states was the correction.
We run both models against the measured values and read their ratio across three
temperatures:

```{math}
:label: eq-diamond-closed
\frac{C_{\text{Einstein}}(\theta_E = 1320)}{C_{\text{Debye}}(\theta_D = 2230)}
= 1.5 \to 0.5 \to 0.002
\quad\text{across } 300/150/75\ \text{K},
```

the 1907 → 1912 story completed. Einstein's fit matched the 300 K point by construction and
collapses below it; Debye with the low-temperature $\theta_D = 2230$ K owns the low-$T$ law, and *undershoots* at 300 K ($C/3R = 0.163$ against the measured $0.245$). The subtlety is
taught as data: the fitted $\theta_D$ drifts with the fitting window (the classic
$\theta_D(T)$ plot) because the true $g(\omega)$ is not a parabola-with-cliff, and the drift is precisely the measured residual between Debye's caricature and the real phonon spectrum, which neutron spectroscopy photographs directly (outward).

### Dimension counts

Nothing in the census argument is specifically three-dimensional, and the 1D chain lets us
test it with no Debye caricature at all: the spectrum is known in closed form, so we sum
the per-oscillator heat capacity of [§7.5](oscillator-at-temperature.ipynb) over every mode of the ring, the $k = 0$ translation
excluded:

```{math}
:label: eq-dimension-counts
\frac{C}{Nk_B} = \frac{1}{N}\sum_{k\neq0}\frac{x_k^2e^{x_k}}{(e^{x_k}-1)^2},
\quad x_k = \frac{\hbar\omega_k}{k_BT}
\qquad\Longrightarrow\qquad
\frac{C}{Nk_BT} \to \frac{\pi}{3}\ \ (1\text{D}),
```

the exact chain's mode sum: $C \propto T$ in one dimension with coefficient exactly $\pi/3$
(from the chain's constant low-$\omega$ density of states), $T^2$ in sheets (graphene's flexural modes complicate the story, one breath), $T^3$ in crystals: one census, dimension merely counting the awake modes. And a finite-size lesson en route: below its softest mode ($\omega_{\min} \approx 2\pi v_s/Na$) a finite chain freezes like a gapped system, the shell-staircase spirit of [§7.9](fermi-gas-zero-temperature.ipynb) in phonon dress, and the reason nanoparticle calorimetry sees size
effects.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Constants and cited material data. Sound speeds and atom densities: Kittel,
# Introduction to Solid State Physics (Ch. 3/5 tables); calorimetric Debye
# temperatures: Ashcroft & Mermin Table 23.3 lineage (Cu 343, Al 428, Pb 105 K;
# diamond low-T value 2230 K). Diamond's measured room-temperature heat capacity:
# Cp(298.15 K) = 6.115 J mol^-1 K^-1 (JANAF tables). Cited, not tuned.
from scipy.constants import (
    hbar as HBAR,
)  # J·s (CODATA; the exact-h derivation is the rule of §7.13, moot here)
from scipy.constants import k as KB_SI  # J/K (exact)

R_GAS = 8.314462618  # J/(mol K) (exact)
SOUND = {  # v_L, v_T (m/s), atom density n (m^-3)
    "Cu": (4760.0, 2325.0, 8.49e28),
    "Al": (6420.0, 3040.0, 6.03e28),
    "Pb": (1960.0, 700.0, 3.30e28),
}
THETA_CAL = {"Cu": 343.0, "Al": 428.0, "Pb": 105.0, "diamond": 2230.0}
THETA_E_DIAMOND = 1320.0  # the Einstein fit of §7.5, reused
CP_DIAMOND_298 = 6.115  # J/(mol K), JANAF
GAMMA_CU_FREE = 0.505  # mJ/(mol K^2): the free-electron gamma of §7.10 for copper (provenance: §7.10)


def chain_modes(N, K=1.0, M=1.0):
    """Normal modes of the N-atom ring: dynamical matrix eigenvalues via numpy.linalg.eigh.

    D = (K/M)(2I − S − S^T) with S = numpy.roll(numpy.eye(N), 1, axis=1) — to the
    letter the tight-binding circulant of §7.12 (one matrix, two physics). eigh returns
    ω^2; the k = 0 translation mode comes back at ±1e-16 with a sign at the mercy
    of round-off (build- and size-dependent), and a naive numpy.sqrt returns NaN
    whenever it lands negative — so frequencies pass through
    numpy.sqrt(numpy.maximum(w2, 0)) (the hazard Exercise 1 demonstrates before
    clamping).

    Parameters
    ----------
    N : int
        Number of atoms on the ring.
    K : float, optional
        Spring constant (default 1, working units).
    M : float, optional
        Atomic mass (default 1).

    Returns
    -------
    numpy.ndarray
        The N mode frequencies, sorted ascending (the first is the zero mode).
    """
    S = np.roll(np.eye(N), 1, axis=1)
    D = (K / M) * (2.0 * np.eye(N) - S - S.T)
    w2 = np.linalg.eigh(D)[0]
    return np.sort(np.sqrt(np.maximum(w2, 0.0)))


def debye_C(t):
    """The universal Debye heat capacity C/3Nk_B = 3t^3 ∫_0^{1/t} x^4 e^x/(e^x−1)^2 dx.

    Nondimensionalized per the standing rule of §7.14 (quad meets physics only after the
    physics is order one). The kernel is written with numpy.expm1 as
    x^4 e^x / expm1(x)^2, and the lower limit is 1e-8 rather than a literal 0:
    the integrand behaves as x^2 there (integrable, harmless), but expm1(0)^2 = 0
    puts a 0/0 at the exact endpoint — the stated cutoff costs ~1e-25 of the
    integral and buys a clean evaluation.

    Parameters
    ----------
    t : float
        Reduced temperature T/θ_D.

    Returns
    -------
    float
        C/3Nk_B.
    """
    kernel = lambda x: x**4 * np.exp(x) / np.expm1(x) ** 2
    val, _ = quad(kernel, 1e-8, 1.0 / t, limit=200)
    return 3.0 * t**3 * val


def einstein_C(t):
    """The Einstein heat capacity per mode of §7.5, C/k_B = x^2 e^{−x}/(1 − e^{−x})^2, x = 1/t.

    Written in the overflow-safe e^{−x} form (x can reach several hundred at the
    low temperatures this notebook visits; the naive e^x form overflows at
    x ≈ 709).

    Parameters
    ----------
    t : float
        Reduced temperature T/θ_E.

    Returns
    -------
    float
        C/k_B per mode (equals C/3Nk_B for the Einstein solid).
    """
    x = 1.0 / t
    em = np.exp(-x)
    return x**2 * em / (1.0 - em) ** 2


def debye_average_speed(vL, vT):
    """The Debye polarization average v_D = [3/(v_L^-3 + 2 v_T^-3)]^{1/3}.

    One longitudinal and two transverse branches contribute their mode counts as
    1/v^3 each — so the average is harmonic in v^3, heavily weighted toward the
    SLOW transverse speeds. Using a single "the" sound speed silently shifts θ_D
    (the stated trap): for copper the naive v_L would give θ_D off by ~80%.

    Parameters
    ----------
    vL : float
        Longitudinal sound speed, m/s.
    vT : float
        Transverse sound speed, m/s.

    Returns
    -------
    float
        v_D in m/s.
    """
    return (3.0 / (vL**-3 + 2.0 * vT**-3)) ** (1.0 / 3.0)


def theta_from_sound(vL, vT, n):
    """The acoustic Debye temperature θ_D = (ħ v_D/k_B)(6π^2 n)^{1/3}.

    The census fixes ω_D by ∫g dω = 3N (ω_D^3 = 6π^2 n v_D^3); the only inputs
    are the two sound speeds (averaged by debye_average_speed) and the atom
    density — no calorimeter anywhere. Exercise 5 compares this against the
    calorimetric θ_D: two apparatus-independent routes to one parameter.

    Parameters
    ----------
    vL : float
        Longitudinal sound speed, m/s.
    vT : float
        Transverse sound speed, m/s.
    n : float
        Atom number density, m^-3.

    Returns
    -------
    float
        θ_D in kelvin.
    """
    vD = debye_average_speed(vL, vT)
    omega_D = vD * (6.0 * np.pi**2 * n) ** (1.0 / 3.0)
    return HBAR * omega_D / KB_SI


def chain_C_exact(T, N):
    """The exact chain heat capacity per atom: the mode sum of eq-dimension-counts.

    C/Nk_B = (1/N) Σ_{k≠0} x^2 e^{−x}/(1 − e^{−x})^2 with x = ω_k/T (working
    units), the k = 0 translation mode EXCLUDED (a free coordinate, not an
    oscillator) and the summand written in the overflow-safe e^{−x} form (x
    reaches ~200 in Exercise 7's freeze demonstration; e^x there would overflow).

    Parameters
    ----------
    T : float
        Temperature in working units (k_B = ħ = 1).
    N : int
        Chain length.

    Returns
    -------
    float
        C/(N k_B).
    """
    k = 2.0 * np.pi * np.arange(1, N) / N  # k = 0 excluded: the free-coordinate rule
    omega = 2.0 * np.abs(np.sin(k / 2.0))
    x = omega / T
    em = np.exp(-x)
    return float(np.sum(x**2 * em / (1.0 - em) ** 2)) / N


def A_coefficient(theta_D):
    """The low-temperature lattice coefficient A = 12π^4 R/(5 θ_D^3), in mJ mol^-1 K^-4.

    The T^3 law's molar prefactor — the number §7.10 invoked for its C/T-vs-T^2
    plot with the derivation deferred to this notebook (Exercise 4 performs the
    comparison against the displayed digit of §7.10).

    Parameters
    ----------
    theta_D : float
        Debye temperature, K.

    Returns
    -------
    float
        A in mJ mol^-1 K^-4.
    """
    return 12.0 * np.pi**4 * R_GAS / (5.0 * theta_D**3) * 1e3


print(f"cited: θ_D(cal) = {THETA_CAL}   (A&M Table 23.3 lineage)")
print(
    f"diamond anchors: θ_E = {THETA_E_DIAMOND:.0f} K (§7.5), Cp(298) = {CP_DIAMOND_298} J/(mol K) (JANAF)"
)

## Exercise 1 — One matrix, two physics

The chain diagonalized, with the circulant of [§7.12](bloch-theorem-band-structure.ipynb) returning as a spring problem. Cite
{eq}`eq-chain-dispersion`.

1. Derive the equations of motion and the dynamical matrix $D = (K/M)(2I - S - S^{\mathsf T})$;
   state the identity with the tight-binding matrix of [§7.12](bloch-theorem-band-structure.ipynb).
2. Diagonalize with `numpy.linalg.eigh` and verify $\omega(k) = 2\sqrt{K/M}|\sin(ka/2)|$ to
   $\sim10^{-15}$ — demonstrating the zero-mode round-off hazard first (the translation
   eigenvalue returns at $\pm10^{-16}$ with build- and size-dependent sign, and naive
   `numpy.sqrt` yields NaN whenever it lands negative) and clamping with
   `numpy.sqrt(numpy.maximum(w2, 0))`.
3. Extract the sound speed from the small-$k$ slope and verify $v_s = a\sqrt{K/M}$; plot the
   dispersion with the sound line tangent.
4. Count (prose + one line): exactly $N$ modes, against the infinity of [§7.14](photon-gas-planck.ipynb): a crystal cannot catastrophe; its cutoff is a census.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    zero_at_roundoff and np.isnan(nan_demo),
    "the zero-mode hazard: the translation eigenvalue sits at round-off with unspecified sign, and naive sqrt fails whenever it lands negative",
    f"zero modes {min(zero_modes.values()):+.1e} … {max(zero_modes.values()):+.1e}",
)
validate.close(
    list(freqs[1:]),
    list(exact[1:]),
    "the chain: one circulant, two physics — eigh lands on 2√(K/M)|sin(ka/2)| on every physical mode",
    atol=1e-12,
)
validate.check(
    freqs[0] < 1e-6,
    "and the clamped zero mode is a √(round-off) artifact, removed by the free-coordinate rule",
    f"freqs[0] = {freqs[0]:.1e}",
)
validate.close(
    v_s,
    1.0,
    "and sound emerges at small k with speed a√(K/M)",
    rtol=1e-3,
)

## Exercise 2 — Quantize, and reckon with Einstein

Each mode an oscillator; the confession recalled; the census argument that resolves it. Cite
{eq}`eq-phonon-quantization`, {eq}`eq-census`.

1. Quantize the modes ([§7.5](oscillator-at-temperature.ipynb) invoked; $\mu = 0$ by the argument of [§7.14](photon-gas-planck.ipynb) in one line) and write
   $U = \sum\hbar\omega(n_B + \tfrac12)$; state honestly what a phonon is.
2. Recall the confession of [§7.5](oscillator-at-temperature.ipynb) quantitatively: evaluate `einstein_C` for diamond at 50 K
   ($\theta_E = 1320$ K) and compare with the $T^3$ behaviour experiment shows.
3. Make the census argument concrete on the chain: count the modes with $\omega < T$ (`numpy.count_nonzero` on the Exercise 1 spectrum) at two temperatures and verify the count doubles when $T$ doubles: $C \propto T^d$ before any integral.
4. State the division (prose): Einstein proved quantization was the point (1907); the spectrum's *shape* at low $\omega$ is what he missed; Debye supplies it (1912). The story is completed in Exercise 6.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    C_einstein_50K < 1e-8,
    "the confession, quantified: Einstein's diamond is exponentially silent at 50 K",
    f"C/3Nk_B = {C_einstein_50K:.1e}",
)
validate.close(
    awake_b / awake_a,
    2.0,
    "why Einstein failed: the acoustic census — awake modes scale with T before any integral",
    rtol=5e-2,
)

## Exercise 3 — The Debye model, and both limits

The universal curve, nondimensionalized per the standing rule. Cite {eq}`eq-debye-model`.

1. Derive $g(\omega) = 3V\omega^2/2\pi^2v_D^3$ with the polarization average
   $v_D = [3/(v_L^{-3}+2v_T^{-3})]^{1/3}$, and fix $\omega_D$ ($\theta_D$) by the census
   $\int g = 3N$.
2. Implement `debye_C(t)` via `scipy.integrate.quad` on $3t^3\int x^4e^x/(e^x-1)^2dx$
   (`numpy.expm1` in the kernel; the stated lower cutoff $10^{-8}$; the nondimensionalization rule of [§7.14](photon-gas-planck.ipynb)
   re-stated in the solution).
3. Verify both limits: $C/3Nk_B = 0.998$ at $t = 5$ (Dulong–Petit: Volume V's law recovered as a limit) and the $T^3$ law with coefficient $4\pi^4/5$ at ratio 1.00000 at $t = 0.02$
   (the integral of [§7.3](statistical-toolkit.ipynb) credited).
4. Plot the universal curve with both limiting laws dashed, and place Cu, Al, Pb, and diamond
   on it at $t = 300\,\text{K}/\theta_D$ — why lead is classical and diamond quantum at room
   temperature, in one sentence.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    [C_t5, ratio_T3],
    [0.998, 1.0000],
    "Debye: Dulong–Petit above, the exact T^3 below — Volume V's law now a limit",
    rtol=2e-3,
)

## Exercise 4 — The coefficient 7.10 used, derived to the digit

[§7.10](fermi-gas-finite-temperature.ipynb) invoked a lattice coefficient with its derivation deferred here; this exercise derives
it and compares, digit for digit. Cite {eq}`eq-A-coefficient`.

1. Derive $A = 12\pi^4R/5\theta_D^3$ from the low-$T$ limit of Exercise 3.
2. Evaluate `A_coefficient(343)` for copper and compare against the number [§7.10](fermi-gas-finite-temperature.ipynb) used,
   verbatim: $0.0482$ mJ mol⁻¹K⁻⁴.
3. Re-assemble the $C/T$-versus-$T^2$ plot of [§7.10](fermi-gas-finite-temperature.ipynb) now fully first-principles ($\gamma$ from
   the free-electron value of [§7.10](fermi-gas-finite-temperature.ipynb), $A$ from here) and confirm $T^* = \sqrt{\gamma/A} = 3.2$ K.
4. Note the practice (prose, one paragraph): a coefficient invoked six notebooks ago and derived today, identical to the displayed digit; anti-redundancy as an auditable habit.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    A_Cu,
    0.0482,
    "the T^3 coefficient, derived: the same figure §7.10 displayed",
    rtol=1e-3,
)
validate.close(
    T_star,
    3.2,
    "and the crossover of §7.10 now stands fully first-principles",
    rtol=3e-2,
)

## Exercise 5 — Two experiments, one number

$\theta_D$ by calorimeter and by loudspeaker — copper agrees with itself to three digits.
Cite {eq}`eq-sound-vs-heat`.

1. Implement `theta_from_sound(vL, vT, n)` with the mandatory polarization average
   `debye_average_speed` (the single-speed trap stated) and the cited inputs.
2. Verify: Cu 342 vs calorimetric 343 K (ratio 1.00); Al 399 vs 428 (0.93); Pb 76 vs 105
   (0.72); plot the bar pairs.
3. Teach the pattern (prose): copper is the harmonic approximation earning its keep; lead's 28% miss is *anharmonicity* (soft, heavy, non-Hookean at accessible temperatures), with thermal expansion named as its everyday face.
4. Weigh the epistemics (prose): two apparatus-independent routes to one parameter is the strongest kind of validation this course can stage: sound and heat agreeing about $\hbar$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    theta_sound["Cu"],
    343.0,
    "copper's Debye temperature, by loudspeaker: the calorimeter's number to three digits",
    rtol=1e-2,
)
validate.close(
    [theta_sound["Al"], theta_sound["Pb"]],
    [399.0, 76.0],
    "aluminum and lead land on their stated acoustic values — the misses are anharmonicity's",
    rtol=3e-2,
)

## Exercise 6 — Diamond, closed honestly

The 1907 → 1912 story completed — including the part where Debye is only a caricature. The
two cited anchors are the Einstein fit of [§7.5](oscillator-at-temperature.ipynb) ($\theta_E = 1320$ K) with the literature low-$T$
Debye value ($\theta_D = 2230$ K), and diamond's measured room-temperature heat capacity
($C_p(298) = 6.115$ J mol⁻¹K⁻¹, JANAF). Cite {eq}`eq-diamond-closed`.

1. Exhibit the Einstein collapse: the ratio `einstein_C`/`debye_C` at 300, 150, and 75 K ($1.5 \to 0.5 \to 0.002$) on a log axis; Debye owns the regime Einstein could not reach.
2. Exhibit the 300 K *undershoot* honestly: Debye with the low-$T$ $\theta_D = 2230$ K gives
   $C/3R = 0.163$ against the measured $0.245$.
3. Quantify the drift: solve `scipy.optimize.brentq` on $C_{\text{Debye}}(298/\theta) =
   C_p^{\text{JANAF}}/3R$ for the *effective* $\theta_D(298)$ and compare with the low-$T$
   2230 K — the fitted $\theta_D$ depends on the fitting window because the true $g(\omega)$
   is not a parabola-with-cliff (the classic $\theta_D(T)$ plot named).
4. Complete the story (prose): Einstein 1907 proved heat capacity is quantum; Debye 1912 got
   the spectrum's low-$\omega$ shape right; the drift is the caricature's measured residual,
   and neutron spectroscopy photographs the real $g(\omega)$ (outward); the anomaly of [§7.5](oscillator-at-temperature.ipynb) is now closed at both ends.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    [ratios_ED[300.0], ratios_ED[75.0]],
    [1.47, 0.002],
    "diamond closed: comparable at 300 K, Einstein 500× silent by 75 K",
    rtol=5e-1,
)
validate.close(
    C_debye_298,
    0.163,
    "and the honest undershoot: the low-T θ_D misses the room-temperature point by a third",
    rtol=5e-2,
)
validate.check(
    1600.0 < theta_eff_298 < 2000.0 and theta_eff_298 < theta_D_dia,
    "the θ_D(T) drift, quantified: two fitting windows, two Debye temperatures",
    f"θ_eff(298 K) = {theta_eff_298:.0f} K vs low-T 2230 K",
)

## Exercise 7 — Dimension counts, and a chain that freezes

The exact 1D mode sum: a $\pi/3$, a general law, and a finite-size lesson. Cite
{eq}`eq-dimension-counts`.

1. Implement `chain_C_exact(T, N)` as the mode sum ($k = 0$ excluded — the free-coordinate
   rule; the overflow-safe $e^{-x}$ form stated).
2. Verify $C/NT \to \pi/3$ above the finite-size gap ($N = 4096$, $t = 0.05$), and derive the
   $\pi/3$ from the chain's constant low-$\omega$ density of states.
3. Exhibit the *freeze*: at $N = 64$ the softest mode sits at $\omega_{\min} \approx 2\pi
   v_s/Na$, and at $T = 0.01$ the chain's heat capacity has collapsed to $\sim10^{-4}$;
   choose windows consciously, and read the freeze as physics (nanoparticle calorimetry, one
   breath; the staircase spirit of [§7.9](fermi-gas-zero-temperature.ipynb)).
4. Generalize (prose + the census): $C \propto T^d$ ($T$ in chains, $T^2$ in sheets with graphene's flexural caveat, $T^3$ in crystals); one argument, dimension merely counting the awake modes.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    C_over_T,
    np.pi / 3,
    "dimension counts: the chain's linear law with its exact π/3 coefficient",
    rtol=1e-2,
)
validate.check(
    C_frozen < 1e-3,
    "and the finite-size freeze: below its softest mode the 64-atom chain goes silent",
    f"C/Nk_B = {C_frozen:.1e} at T = 0.01",
)

## Exercise 8 — The audit clears

This notebook was an audit, and every line cleared. The oscillator that [§7.5](oscillator-at-temperature.ipynb) warmed became a
crystal's normal mode; the matrix that hopped electrons in [§7.12](bloch-theorem-band-structure.ipynb) swayed atoms here; the
$\mu = 0$ of [§7.14](photon-gas-planck.ipynb) carried over in a sentence; and the integral [§7.3](statistical-toolkit.ipynb) computed before any of this existed delivered, at last, the $T^3$ coefficient that [§7.10](fermi-gas-finite-temperature.ipynb) had used on trust: derived today, identical to the displayed digit. Dulong–Petit, which Volume V asserted, is now a
high-temperature limit; Einstein's diamond, which [§7.5](oscillator-at-temperature.ipynb) could only match at one point, is closed across the range, including the honest part, where Debye's one-parameter caricature drifts and the drift itself is data. And the quietest result may be the strongest: copper's
Debye temperature, measured once with a calorimeter and once with sound, agreeing to three digits: heat and acoustics cross-examining each other about Planck's constant and telling the same story.

There is something satisfying about a model that knows its own size. The photon gas needed quantum mechanics to escape an infinity; the crystal never faced one: it has $3N$ modes because it has $N$ atoms, and Debye's famous cutoff is just the model refusing to invent
degrees of freedom it was never given. Half of good physics is bookkeeping done with
conviction.

One boson gas remains, the one whose particle number is protected: when *its* ceiling bites,
the gas does not glow or hum — it condenses ([§7.17](bose-einstein-condensation.ipynb)).

## Notebook summary

Movement IV's third notebook: the crystal as a photon gas with a speed limit and a headcount.

- **One matrix, two physics** {eq}`eq-chain-dispersion`: the ring's dynamical matrix is
  the tight-binding circulant of [§7.12](bloch-theorem-band-structure.ipynb); `eigh` lands on $2\sqrt{K/M}|\sin(ka/2)|$ at $10^{-12}$ (gated) with sound at small $k$ (gated), after the zero-mode hazard is demonstrated and clamped (gated). Exactly $N$ modes: the cutoff is a census, and no catastrophe is possible.
- **Quantization and the census** {eq}`eq-phonon-quantization`, {eq}`eq-census`: each mode is
  the oscillator of [§7.5](oscillator-at-temperature.ipynb) at $\mu = 0$ (the argument of [§7.14](photon-gas-planck.ipynb)); Einstein's diamond is $10^{-9}$-silent at 50 K (gated) because his solid has nothing soft to offer, while the chain's awake-mode count doubles with $T$ (gated): $C \propto T^d$ before any integral.
- **Debye, both limits** {eq}`eq-debye-model`: the universal curve (nondimensionalized per the rule of [§7.14](photon-gas-planck.ipynb), `expm1` kernel, stated cutoff) reaches 0.998 at $t = 5$ (Dulong–Petit as a *limit*) and lands on $(4\pi^4/5)t^3$ at ratio 1.00000 (both gated; the integral of [§7.3](statistical-toolkit.ipynb),
  third use).
- **The coefficient, derived** {eq}`eq-A-coefficient`: $A(343\,\text{K}) = 0.0482$ mJ mol⁻¹K⁻⁴, the displayed digit of [§7.10](fermi-gas-finite-temperature.ipynb) (gated), and $T^* = 3.2$ K now first-principles (gated).
- **Two experiments, one number** {eq}`eq-sound-vs-heat`: acoustic $\theta_D$ from cited $v_L, v_T, n$ against calorimetric: Cu 342/343 (gated at 1%), Al 0.93, Pb 0.72 (gated), with lead's miss taught as anharmonicity and the single-speed trap stated.
- **Diamond, closed honestly** {eq}`eq-diamond-closed`: Einstein/Debye = 1.5 → 0.002 across
  300–75 K (gated); the low-$T$ $\theta_D = 2230$ K undershoots the JANAF 298 K point
  ($0.163$ vs $0.245$, gated); the window-fitted $\theta_D \approx 1860$ K (gated as a band)
  exhibits the $\theta_D(T)$ drift as the caricature's measured residual.
- **Dimension counts** {eq}`eq-dimension-counts`: the exact mode sum plateaus at $\pi/3$ (gated); the 64-atom chain freezes below $2\pi v_s/Na$ (gated): the staircase spirit of [§7.9](fermi-gas-zero-temperature.ipynb) in phonon dress.

Next door, the boson gas whose number is conserved runs out of room.

## Outlook

- **BEC ([§7.17](bose-einstein-condensation.ipynb)).** The conserved-number boson gas and its saturating ceiling: when this one
  bites, the gas condenses.
- **Real phonon spectra.** Neutron and X-ray spectroscopy (where $g(\omega)$ is photographed);
  anharmonicity, thermal expansion, and phonon–phonon scattering (outward horizons, named).
- **Transport.** Thermal conductivity: the phonon gas as a transport problem (outward, named).
- **Low dimensions.** Graphene's flexural branch and the $T^2$ story's fine print;
  nanoparticle calorimetry and the finite-size freeze (outward, named).
- **Cross-reference** [§7.5](oscillator-at-temperature.ipynb) (the confession, resolved), [§7.10](fermi-gas-finite-temperature.ipynb) (the coefficient, now derived),
  [§7.12](bloch-theorem-band-structure.ipynb) (the circulant; the counting), [§7.14](photon-gas-planck.ipynb) ($\mu = 0$; the nondimensionalization rule; the
  catastrophe contrast), [§7.3](statistical-toolkit.ipynb) (the integral, thrice), [§5.5](../05-classical-stat-mech/ergodicity.ipynb) (Dulong–Petit, now a limit).

In [ ]:
from ecp.style import footer

footer()